# Week 2

Original Code

generate_full_space_tree.py

In [ ]:
from collections import deque
import pydot
import argparse
import os

# Set it to bin folder of graphviz
os.environ["PATH"] += os.pathsep + '/opt/homebrew/bin'

options = [(1,0), (0, 1), (1, 1), (0, 2), (2, 0)] # trạng thái của số người và số quỷ khi di chuyển qua lại 2 bờ, mỗi lần chỉ chỗ 1 hoặc 2 chỗ

Parent = dict()
graph = pydot.Dot(graph_type = 'graph', strict = False, bgcolor='#fff3af',
                  label='fig: Missionaries and Cannibal State Space Tree')
# To track node (theo dõi)
i = 0
arg = argparse.ArgumentParser() # xử lý các tham số truyền vào khi chạy trương trình từ command line (terminal)
arg.add_argument('-d', '--depth', required = False, # xác định độ sâu tối đa của đồ thị trong không gian trạng thái
                 help = 'Maximum depth upto which ypu want to generate Space State Tree')

args = vars(arg.parse_args()) # phân tích và lấy các đối số từ dòng lệnh và đưa chúng vào một dictionary để dễ dàng truy xuất

max_depth = int(args.get('depth', 20))

def is_valid_move(number_missionaries, number_cannibals): # kiểm tra số lượng nhà truyền giáo và ăn thịt người có hợp lệ không 
    """
    Check if number constraints are satisfied
    """
    return (0 <= number_missionaries <= 3) and (0 <= number_cannibals <= 3)

def write_image(file_name = 'state_space'): # lưu đồ thị vào file
    try:
        graph.write_png(f"{file_name}_{max_depth}.png")
    except Exception as e:
        print('Error while writing file', e)
        print(f"File {file_name}_{max_depth}.png successfully written.")

def draw_edge(number_missionaries, number_cannibals, side, depth_level, node_num): # vẽ các cạnh nối giữa các nút
    # Các tham số của hàm đại diện cho trạng thái của một nút trong đồ thị:
    # - `number_missionaries`: Số lượng nhà truyền giáo ở một bên bờ sông
    # - `number_cannibals`: Số lượng kẻ ăn thịt người ở một bên bờ sông
    # - `side`: Bờ sông mà thuyền đang ở (ví dụ: 0 là bờ trái, 1 là bờ phải)
    # - `depth_level`: Độ sâu của nút trong cây không gian trạng thái
    # - `node_num`: Số thứ tự của nút (để phân biệt các trạng thái khác nhau tại cùng một độ sâu)
    u, v = None, None
    if Parent[(number_missionaries, number_cannibals, side, depth_level, node_num)] is not None: # kiểm tra nút cha của nút hiện tại có tồn tại không
        # Nếu nút cha tồn tại, tức là nút hiện tại không phải là nút gốc.
        # Tạo nút `u` là nút cha của nút hiện tại.
        u = pydot.Node(str(Parent[(number_missionaries, number_cannibals, side, depth_level, node_num)]),
                       label = str(Parent[(number_missionaries, number_cannibals, side, depth_level, node_num)][:3]))
        graph.add_node(u) # thêm nút 'u' vào đồ thị

        # tạo nút `v` là nút hiện tại
        v = pydot.Node(str((number_missionaries, number_cannibals, side, depth_level, node_num)),
                       label= str((number_missionaries, number_cannibals, side ))) # depth_level, node_num
        graph.add_node(v)

        # tạo cạnh nối từ nút `u` đến nút `v`
        edge = pydot.Edge(str(Parent[(number_missionaries, number_cannibals, side, depth_level, node_num)]),
                           str((number_missionaries, number_cannibals, side, depth_level, node_num)), dir = 'forward')
        graph.add_edge(edge) 
    else:
        # For start node
        # nếu nút hiện tại là nút gốc
        # tạo nút `v`
        v = pydot.Node(str((number_missionaries, number_cannibals, side, depth_level, node_num)),
                       label = str((number_missionaries, number_cannibals, side)))
        graph.add_node(v)
    return u, v

def is_start_state(number_missionaries, number_cannibals, side): # trạng thái ban đầu ở bờ bên trái (3 nhà truyền giáo, 3 ăn thịt người, thuyền ở bờ bên trái)
    return (number_missionaries, number_cannibals, side) == (3, 3, 1)

def is_goal_state(number_missionaries, number_cannibals, side): # trạng thái đích ở bờ bên trái (0 nhà truyền giáo, 0 ăn thịt người, thuyền ở bờ bên phải)
    return (number_missionaries, number_cannibals, side) == (0, 0, 0)

def number_of_cannibals_exceeds(number_missionaries, number_cannibals): 
    # kiểm tra xem số ăn thịt người có nhiều hơn số nhà truyền giáo hay không
    # ở cả hai bên bờ
    number_missionaries_right = 3 - number_missionaries                   
    number_cannibals_right = 3 - number_cannibals
    return number_missionaries > 0 and number_cannibals > number_missionaries\
    or (number_missionaries_right > 0 and number_cannibals_right > number_missionaries_right)


def generate(): # tạo đồ thị theo độ sâu đã nhập trong không gian trạng thái
    global i
    q = deque()
    node_num = 0
    q.append((3, 3, 1, 0, node_num)) # thêm trạng thái bắt đầu vào hàng đợi

    Parent[(3, 3, 1, 0, node_num)] = None

    while q: # nếu hàng đợi không rỗng
        number_missionaries, number_cannibals, side, depth_level, node_num = q.popleft() # lấy ra nút ở đầu hàng đợi
        # print(number_missionaries, number_cannibals)
        # # Draw Edge from u -> v
        # Where u = Parent(v)
        # and v = (number_missionaries, number_cannibals, side, depth_level)

        u, v = draw_edge(number_missionaries, number_cannibals, side, depth_level, node_num) # tạo các nút và cạnh
        # u = v

        if is_start_state(number_missionaries, number_cannibals, side):
            v.set_style("filled")
            v.set_fillcolor("blue") #fontcolor
        elif is_goal_state(number_missionaries, number_cannibals, side):
            v.set_style("filled")
            v.set_fillcolor("green")
            continue
            # return True
        elif number_of_cannibals_exceeds(number_missionaries, number_cannibals): # nếu không thoả điều kiện (ăn thịt người > nhà truyền giáo)
            v.set_style("filled") 
            v.set_fillcolor("red")
            continue
        else:
            v.set_style("filled") # thoả điều kiện
            v.set_fillcolor("orange")

        if depth_level == max_depth: # Khi đạt đến độ sâu tối đa (max_depth), hàm sẽ dừng mở rộng và trả về True
            return True

        op = -1 if side == 1 else 1 # op là một giá trị cho biết thuyền đang ở bờ nào (1 là bờ đích, -1 là bờ bắt đầu).
        can_be_expanded = False

        i = node_num
        for x, y in options: 
            next_m, next_c, next_s = number_missionaries + op * x, number_cannibals + op * y, int(not side) # cập nhật trạng thái cho bờ bên trái sau khi di chuyển
            # in(not side) đổi vị trí nếu side == 1 thì trả về False <=> int(False) == 0 và ngược lại
            
            # Điều kiện đảm bảo rằng trạng thái mới không phải là nút cha của trạng thái hiện tại (tránh quay lại trạng thái cũ).
            if Parent[(number_missionaries, number_cannibals, side, depth_level, node_num)] is None or \
               (next_m, next_c, next_s) != Parent[(number_missionaries, number_cannibals, side, depth_level, node_num)][:3]:

                # nếu trạng thái di chuyển hợp lệ, trạng thái mới sẽ được thêm vào hàng đợi q, đồng thời lưu lại quan hệ cha-con trong Parent.
                if is_valid_move(next_m, next_c):
                    can_be_expanded = True
                    i += 1
                    q.append((next_m, next_c, next_s, depth_level + 1, i))
                    # keep track of parent
                    Parent[(next_m, next_c, next_s, depth_level + 1, i)] = \
                        (number_missionaries, number_cannibals, side, depth_level, node_num)

        if not can_be_expanded: # Nếu không có di chuyển hợp lệ nào, nút sẽ được đặt thành màu xám, cho biết nút này không thể mở rộng thêm.
            v.set_style("filled")
            v.set_fillcolor("gray")

    return False

if __name__ == "__main__": # Nếu gọi hàm generate() thành công, hàm write_image() sẽ được gọi để lưu đồ thị đã vẽ dưới dạng hình ảnh.
    if generate():
        write_image()

solve.py

In [ ]:
import os
import emoji # type: ignore
import pydot # type: ignore
import random
from collections import deque

# Set it to bin folder of graphviz
os.environ["PATH"] += os.pathsep + '/opt/homebrew/bin'

# Dictionaries to backtrack solution nodes
# Parent stores parent of (m, c, s)
# Move stores (x, y), i.e. number of missionaries,
# cannibals to be moved from left to right for particular state
# node_list stores pydot.Node object for particular state (m, c, s) so that we can color the solution nodes
Parent, Move, node_list = dict(), dict(), dict()

class Solution():

    def __init__(self):
        # Start state (3M, 3C, Left)
        # Goal state (0M, 0C, Right)
        # Each state gives the number of missionaries and cannibals on the left side

        self.start_state = (3, 3, 1)
        self.goal_state = (0, 0, 0)
        self.options = [(1, 0), (0, 1), (1, 1), (0, 2), (2, 0)]

        self.boat_side = ["Right", "Left"]

        self.graph = pydot.Dot(graph_type='graph', bgcolor="#ffffaf",
                               label="Fig: Missionaries and Cannibal State Space Tree", fontcolor="red", fontsize="24")
        self.visited = {}
        self.solved = False

    def is_valid_move(self, number_missionaries, number_cannibals):
        """
        Checks if number constraints are satisfied
        """
        return (0 <= number_missionaries <= 3) and (0 <= number_cannibals <= 3)

    def is_goal_state(self, number_missionaries, number_cannibals, side):
        return (number_missionaries, number_cannibals, side) == self.goal_state

    def is_start_state(self, number_missionaries, number_cannibals, side):
        return (number_missionaries, number_cannibals, side) == self.start_state

    def number_of_cannibals_exceeds(self, number_missionaries, number_cannibals):
        number_missionaries_right = 3 - number_missionaries
        number_cannibals_right = 3 - number_cannibals
        return (number_missionaries > 0 and number_cannibals > number_missionaries) \
               or (number_missionaries_right > 0 and number_cannibals_right > number_missionaries_right)

    def write_image(self, file_name="state_space.png"):
        try:
            self.graph.write_png(file_name)
        except Exception as e:
            print("Error while writing file", e)
        print(f"File {file_name} successfully written.")

    def solve(self, solve_method="dfs"):
        self.visited = dict()
        Parent[self.start_state] = None
        Move[self.start_state] = None
        node_list[self.start_state] = None

        return self.dfs(*self.start_state, 0) if solve_method == "dfs" else self.bfs()
    
    def draw_legend(self): # tạo chú thích cho đồ thị 
        """
        Utility method to draw legend on graph if legend flag is ON
        """
        graphlegend = pydot.Cluster(graph_name="legend", label="Legend", fontsize="20", color="gold",
                                    fontcolor="blue", style="filled", fillcolor="#f4f4f4")

        node1 = pydot.Node("1", style="filled", fillcolor="blue", label="Start Node", fontcolor="white", width="2", fixedsize="true")
        graphlegend.add_node(node1)

        node2 = pydot.Node("2", style="filled", fillcolor="red", label="Killed Node", fontcolor="black", width="2", fixedsize="true")
        graphlegend.add_node(node2)

        node3 = pydot.Node("3", style="filled", fillcolor="yellow", label="Solution nodes", width="2", fixedsize="true")
        graphlegend.add_node(node3)

        node4 = pydot.Node("4", style="filled", fillcolor="gray", label="Can't be expanded", width="2", fixedsize="true")
        graphlegend.add_node(node4)

        node5 = pydot.Node("5", style="filled", fillcolor="green", label="Goal node", width="2", fixedsize="true")
        graphlegend.add_node(node5)

        node7 = pydot.Node("7", style="filled", fillcolor="gold", label="Node with child", width="2", fixedsize="true")
        graphlegend.add_node(node7)

        description = "Each node (m, c, b) represents a \nstate where 'm' is the number of\\a missionaries,\\n the cannibals \\ and 'b' the side of the boat" + \
                  "\n where 'm' represents the left \nside and '0' the right side \\n \nOur objective is to reach goal state (0, 0, 0)\n \n" + \
                  "from start state (3, 3, 1) by some \noperators i.e {(0, 1), (0, 2), (1, 0), (1, 1), (2, 0)}.\n" + \
                  "In these operators \\n represents the number of missionaries and \\ cannibals to be moved from left to right \\n if c = 1 and vice versa"
    
        node6 = pydot.Node("6", style="filled", fillcolor="gold", label=description, shape="plaintext", fontsize="20", fontcolor="red")
        graphlegend.add_node(node6)

        self.graph.add_subgraph(graphlegend)

        self.graph.add_edge(pydot.Edge(node1, node2, style="invis"))
        self.graph.add_edge(pydot.Edge(node2, node3, style="invis"))
        self.graph.add_edge(pydot.Edge(node3, node4, style="invis"))
        self.graph.add_edge(pydot.Edge(node4, node5, style="invis"))
        self.graph.add_edge(pydot.Edge(node5, node6, style="invis"))
        self.graph.add_edge(pydot.Edge(node6, node7, style="invis"))

    def draw(self, number_missionaries_left, number_cannibals_left, number_missionaries_right, number_cannibals_right): 
        """
        Draw state on console using emojis
        """
        left_m = emoji.emojize(":old_man: " * number_missionaries_left)
        left_c = emoji.emojize(":ogre: " * number_cannibals_left)
        right_m = emoji.emojize(":old_man: " * number_missionaries_right)
        right_c = emoji.emojize(":ogre: " * number_cannibals_right)

        print("({}{}{}{}{}{})".format(left_m, left_c, " " * (14 - len(left_m) - len(left_c)), \
                              "_" * 40, " " * (12 - len(right_m) - len(right_c)) + right_m, right_c)) # sửa lại để hiển thị đầy đủ , thêm một {}
        print("")

    def show_solution(self): # hiển thị quá trình giải 
        # Recursively start from Goal State
        # And find parent until start state is reached

        state = self.goal_state
        path, steps, nodes = [], [], [] # Các danh sách rỗng để lưu trữ đường đi, các bước di chuyển, và các nút tương ứng.

        while state is not None:
            path.append(state)
            steps.append(Move[state])
            nodes.append(node_list[state])

            state = Parent[state]

        steps, nodes = steps[::-1], nodes[::-1] # đảo ngược từ goal_state lên để có các bước di chuyển và các nút

        number_missionaries_left, number_cannibals_left = 3, 3
        number_missionaries_right, number_cannibals_right = 0, 0

        print("*" * 60)
        self.draw(number_missionaries_left=number_missionaries_left, number_cannibals_left=number_cannibals_left,
                number_missionaries_right=number_missionaries_right, number_cannibals_right=number_cannibals_right)

        for i, ((number_missionaries, number_cannibals, side), node) in enumerate(zip(steps[1:], nodes[1:])):

            if node.get_label() != str(self.start_state):
                node.set_style("filled")
                node.set_fillcolor("yellow")

            print(f"Step {i + 1}: Move {number_missionaries} missionaries and {number_cannibals} "
                  f"cannibals from {self.boat_side[side]} to {self.boat_side[int(not side)]}.")

            op = -1 if side == 1 else 1

            number_missionaries_left = number_missionaries_left + op * number_missionaries
            number_cannibals_left = number_cannibals_left + op * number_cannibals

            number_missionaries_right = number_missionaries_right - op * number_missionaries
            number_cannibals_right = number_cannibals_right - op * number_cannibals

            self.draw(number_missionaries_left=number_missionaries_left, number_cannibals_left=number_cannibals_left,
                     number_missionaries_right=number_missionaries_right, number_cannibals_right=number_cannibals_right)

        print("Congratulations!!! You have solved the problem")
        print("*" * 60)

    def draw_edge(self, number_missionaries, number_cannibals, side, depth_level):
        u, v = None, None
        if Parent[(number_missionaries, number_cannibals, side)] is not None:
            u = pydot.Node(str(Parent[(number_missionaries, number_cannibals, side)] + (depth_level - 1, )),
                           label=str(Parent[(number_missionaries, number_cannibals, side)]))
            self.graph.add_node(u)

            v = pydot.Node(str((number_missionaries, number_cannibals, side, depth_level)),
                           label=str((number_missionaries, number_cannibals, side)))
            self.graph.add_node(v)

            edge = pydot.Edge(str(Parent[(number_missionaries, number_cannibals, side)] + (depth_level - 1, )),
                          str((number_missionaries, number_cannibals, side, depth_level)), dir='forward')
            self.graph.add_edge(edge)
        else:
            # For start node
            v = pydot.Node(str((number_missionaries, number_cannibals, side, depth_level)),
                           label=str((number_missionaries, number_cannibals, side)))
            self.graph.add_node(v)
        return u, v

    def bfs(self):
        q = deque()
        q.append(self.start_state + (0, )) # thêm trạng thái bắt đầu vào hàng đợi
        self.visited[self.start_state] = True # đánh dấu trạng thái bắt đầu là đã được khám phá

        while q:
            number_missionaries, number_cannibals, side, depth_level = q.popleft() # lấy ra nút ở đầu hàng đợi
            # Draw Edge from u -> v
            # Where u = Parent[v]
            # and v = (number_missionaries, number_cannibals, side, depth_level)
            u, v = self.draw_edge(number_missionaries, number_cannibals, side, depth_level) # # tạo các nút và cạnh

            if self.is_start_state(number_missionaries, number_cannibals, side):
                v.set_style("filled")
                v.set_fillcolor("blue")
                v.set_fontcolor("white")
            elif self.is_goal_state(number_missionaries, number_cannibals, side):
                v.set_style("filled")
                v.set_fillcolor("green")
                return True # nếu nút là trạng thái đích
            elif self.number_of_cannibals_exceeds(number_missionaries, number_cannibals):
                v.set_style("filled")
                v.set_fillcolor("red")
                continue
            else:
                v.set_style("filled")
                v.set_fillcolor("orange")

            op = -1 if side == 1 else 1 # op là một giá trị cho biết thuyền đang ở bờ nào (1 là bờ đích, -1 là bờ bắt đầu).

            can_be_expanded = False # Biến này được sử dụng để kiểm tra xem trạng thái hiện tại có thể được mở rộng không

            for x, y in self.options:
                next_m, next_c, next_s = number_missionaries + op * x, number_cannibals + op * y, int(not side) # cập nhật trạng thái cho bờ bên trái sau khi di chuyển
            # in(not side) đổi vị trí nếu side == 1 thì trả về False <=> int(False) == 0 và ngược lại
            
                if (next_m, next_c, next_s) not in self.visited: # kiểm tra xem trạng thái mới này có được khám phá chưa (thoả nếu chưa đc khám phá)
                    if self.is_valid_move(next_m, next_c): # kiểm tra số lượng có hợp lệ không
                        can_be_expanded = True # nếu các điều kiện trên thoả thì nút này có thể được mở rộng
                        self.visited[(next_m, next_c, next_s)] = True # đánh dấu nút đã được khám phá
                        q.append((next_m, next_c, next_s, depth_level + 1)) # thêm nút vào hàng đợi

                        # Keep track of parent and corresponding move
                        Parent[(next_m, next_c, next_s)] = (number_missionaries, number_cannibals, side)
                        Move[(next_m, next_c, next_s)] = (x, y, side)
                        node_list[(next_m, next_c, next_s)] = v

            if not can_be_expanded: 
                v.set_style("filled")
                v.set_fillcolor("gray")

        return False
    
    def dfs(self, number_missionaries, number_cannibals, side, depth_level):
        self.visited[(number_missionaries, number_cannibals, side)] = True

        # Draw Edge from u -> v
        # Where u = Parent[y]
        u, v = self.draw_edge(number_missionaries, number_cannibals, side, depth_level)

        if self.is_start_state(number_missionaries, number_cannibals, side):
            v.set_style("filled")
            v.set_fillcolor("blue")
        elif self.is_goal_state(number_missionaries, number_cannibals, side):
            v.set_style("filled")
            v.set_fillcolor("green")
            return True
        elif self.number_of_cannibals_exceeds(number_missionaries, number_cannibals): 
            v.set_style("filled")
            v.set_fillcolor("red")
            return False
        else:
            v.set_style("filled")
            v.set_fillcolor("orange")

        solution_found = False
        operation = -1 if side == 1 else 1
        can_be_expanded = False

        for x, y in self.options:
            next_m, next_c, next_s = number_missionaries + operation * x, number_cannibals + operation * y, int(not side)

            if (next_m, next_c, next_s) not in self.visited:
                if self.is_valid_move(next_m, next_c):
                    can_be_expanded = True
                    # Keep track of Parent state and corresponding move
                    Parent[(next_m, next_c, next_s)] = (number_missionaries, number_cannibals, side)
                    Move[(next_m, next_c, next_s)] = (x, y, side)
                    node_list[(next_m, next_c, next_s)] = v

                    solution_found = (solution_found or self.dfs(next_m, next_c, next_s, depth_level + 1))

                    if solution_found:
                        return True

        if not can_be_expanded:
            v.set_style("filled")
            v.set_fillcolor("gray")

        self.solved = solution_found
        return solution_found

main.py

In [ ]:
from solve import Solution
import argparse
import itertools

arg = argparse.ArgumentParser()
arg.add_argument("-m", "--method", required=False, help="Specify which method to use")
arg.add_argument("-l", "--legend", required=False, help="Specify if you want to display legend on graph")

args = vars(arg.parse_args())

solve_method = args.get("method", "bfs")
legend_flag = args.get("legend", False)

def main():
    s = Solution()

    if(s.solve(solve_method)):
        # Display Solution on console
        s.show_solution()

        output_file_name = f"{solve_method}"
        # Draw legend if legend_flag is set
        if legend_flag:
            if legend_flag[0].upper() == 'T' :
                output_file_name += "_legend.png"
                s.draw_legend()
            else:
                output_file_name += ".png"
        else:
            output_file_name += ".png"

        # Write State space tree
        s.write_image(output_file_name)
    else:
        raise Exception("No solution found")

if __name__ == "__main__":
    main()